In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('../data/raw/city_hour.csv')

In [ ]:
# Check missing values in the target columns
print("Missing AQI values:", df["AQI"].isna().sum())
print("Missing AQI_Bucket values:", df["AQI_Bucket"].isna().sum())

# Examine the distribution of AQI categories
print("\nAQI_Bucket distribution:")
print(df["AQI_Bucket"].value_counts(dropna=False))

# Inspect numerical AQI statistics
print("\nAQI summary statistics:")
print(df["AQI"].describe())

Missing AQI values: 129080
Missing AQI_Bucket values: 129080

AQI_Bucket distribution:
AQI_Bucket
Moderate        198991
Satisfactory    189434
NaN             129080
Poor             66654
Very Poor        57455
Good             38611
Severe           27650
Name: count, dtype: int64

AQI summary statistics:
count    578795.000000
mean        166.413500
std         162.112729
min           8.000000
25%          79.000000
50%         116.000000
75%         208.000000
max        3133.000000
Name: AQI, dtype: float64


In [ ]:
import numpy as np
import pandas as pd

# Define the expected category for each numerical AQI value
conditions = [
    df["AQI"].between(0, 50),
    df["AQI"].between(51, 100),
    df["AQI"].between(101, 200),
    df["AQI"].between(201, 300),
    df["AQI"].between(301, 400),
    df["AQI"] > 400
]

categories = [
    "Good",
    "Satisfactory",
    "Moderate",
    "Poor",
    "Very Poor",
    "Severe"
]

df["Expected_AQI_Bucket"] = np.select(
    conditions,
    categories,
    default="Invalid_or_Missing"
)

# Identify rows where the recorded and expected categories differ
labelled = df["AQI"].notna() & df["AQI_Bucket"].notna()

mismatches = df.loc[
    labelled
    & (df["AQI_Bucket"] != df["Expected_AQI_Bucket"])
]

print("Rows with both AQI and AQI_Bucket:", labelled.sum())
print("Category mismatches:", len(mismatches))

print("\nMismatch examples:")
print(
    mismatches[
        ["City", "Datetime", "AQI", "AQI_Bucket", "Expected_AQI_Bucket"]
    ].head(20)
)

print("\nAQI values outside the expected 0–500 range:")
print(
    df.loc[
        df["AQI"].notna() & ~df["AQI"].between(0, 500),
        ["City", "Datetime", "AQI", "AQI_Bucket"]
    ].head(20)
)

Rows with both AQI and AQI_Bucket: 578795
Category mismatches: 0

Mismatch examples:
Empty DataFrame
Columns: [City, Datetime, AQI, AQI_Bucket, Expected_AQI_Bucket]
Index: []

AQI values outside the expected 0–500 range:
          City             Datetime     AQI AQI_Bucket
716  Ahmedabad  2015-01-30 21:00:00   953.0     Severe
717  Ahmedabad  2015-01-30 22:00:00   953.0     Severe
718  Ahmedabad  2015-01-30 23:00:00   953.0     Severe
719  Ahmedabad  2015-01-31 00:00:00   953.0     Severe
720  Ahmedabad  2015-01-31 01:00:00   953.0     Severe
721  Ahmedabad  2015-01-31 02:00:00   953.0     Severe
722  Ahmedabad  2015-01-31 03:00:00   953.0     Severe
723  Ahmedabad  2015-01-31 04:00:00   953.0     Severe
724  Ahmedabad  2015-01-31 05:00:00   608.0     Severe
739  Ahmedabad  2015-01-31 20:00:00   518.0     Severe
740  Ahmedabad  2015-01-31 21:00:00   539.0     Severe
741  Ahmedabad  2015-01-31 22:00:00   809.0     Severe
742  Ahmedabad  2015-01-31 23:00:00  1217.0     Severe
743  Ahme

In [ ]:
# Count AQI values outside the expected 0–500 range
below_zero = df["AQI"].notna() & (df["AQI"] < 0)
above_500 = df["AQI"].notna() & (df["AQI"] > 500)

print("AQI values below 0:", below_zero.sum())
print("AQI values above 500:", above_500.sum())

print("\nPercentage of recorded AQI values above 500:")
print(round(above_500.sum() / df["AQI"].notna().sum() * 100, 3), "%")

print("\nMost frequent AQI values above 500:")
print(df.loc[above_500, "AQI"].value_counts().head(15))

print("\nCities with the most AQI values above 500:")
print(df.loc[above_500, "City"].value_counts().head(15))

AQI values below 0: 0
AQI values above 500: 11135

Percentage of recorded AQI values above 500:
1.924 %

Most frequent AQI values above 500:
AQI
507.0    123
516.0    114
510.0    106
508.0    100
504.0     93
620.0     90
537.0     88
503.0     86
501.0     83
524.0     82
528.0     82
538.0     79
511.0     78
506.0     77
532.0     75
Name: count, dtype: int64

Cities with the most AQI values above 500:
City
Ahmedabad     7681
Delhi         1308
Gurugram       596
Patna          569
Lucknow        394
Jorapokhar     153
Hyderabad      140
Talcher        116
Amritsar       100
Guwahati        63
Jaipur          15
Name: count, dtype: int64


In [ ]:
# Summarize AQI values above 500 by city
extreme_aqi = df[df["AQI"] > 500]

city_summary = (
    extreme_aqi.groupby("City")["AQI"]
    .agg(
        count="count",
        minimum="min",
        median="median",
        maximum="max"
    )
    .sort_values("count", ascending=False)
)

print("AQI > 500 records by city:")
print(city_summary.to_string())

print("\nTotal records:", len(extreme_aqi))
print("Cities affected:", extreme_aqi["City"].nunique())

AQI > 500 records by city:
            count  minimum  median  maximum
City                                       
Ahmedabad    7681    501.0   770.0   3133.0
Delhi        1308    501.0   542.0    762.0
Gurugram      596    501.0   551.0    966.0
Patna         569    501.0   538.0    655.0
Lucknow       394    501.0   536.5    761.0
Jorapokhar    153    501.0   563.0    655.0
Hyderabad     140    504.0   546.5    877.0
Talcher       116    501.0   526.0    620.0
Amritsar      100    501.0   755.0   1052.0
Guwahati       63    503.0   867.0   1109.0
Jaipur         15    506.0   534.0    556.0

Total records: 11135
Cities affected: 11


In [ ]:
station_df=pd.read_csv('../data/raw/station_hour.csv')
print("Dataset shape:", station_df.shape)
print("\nColumns:")
print(station_df.columns.tolist())

print("\nFirst 5 rows:")
display(station_df.head())

print("\nData types and non-null counts:")
station_df.info()

C:\Users\hp\AppData\Local\Temp\ipykernel_21780\2698759093.py:1: DtypeWarning: Columns (15: AQI_Bucket) have mixed types. Specify dtype option on import or set low_memory=False.
  station_df=pd.read_csv('../data/raw/station_hour.csv')


Dataset shape: (2589083, 16)

Columns:
['StationId', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']

First 5 rows:


,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,AP001,2017-11-24 17:00:00,60.50,98.00,2.35,30.80,18.25,8.50,0.1,11.85,126.40,0.1,6.10,0.10,NaN,NaN
1,AP001,2017-11-24 18:00:00,65.50,111.25,2.70,24.20,15.07,9.77,0.1,13.17,117.12,0.1,6.25,0.15,NaN,NaN
2,AP001,2017-11-24 19:00:00,80.00,132.00,2.10,25.18,15.15,12.02,0.1,12.08,98.98,0.2,5.98,0.18,NaN,NaN
3,AP001,2017-11-24 20:00:00,81.50,133.25,1.95,16.25,10.23,11.58,0.1,10.47,112.20,0.2,6.72,0.10,NaN,NaN
4,AP001,2017-11-24 21:00:00,75.25,116.00,1.43,17.48,10.43,12.03,0.1,9.12,106.35,0.2,5.75,0.08,NaN,NaN



Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 2589083 entries, 0 to 2589082
Data columns (total 16 columns):
 #   Column      Dtype  
---  ------      -----  
 0   StationId   str    
 1   Datetime    str    
 2   PM2.5       float64
 3   PM10        float64
 4   NO          float64
 5   NO2         float64
 6   NOx         float64
 7   NH3         float64
 8   CO          float64
 9   SO2         float64
 10  O3          float64
 11  Benzene     float64
 12  Toluene     float64
 13  Xylene      float64
 14  AQI         float64
 15  AQI_Bucket  str    
dtypes: float64(13), str(3)
memory usage: 316.1 MB


In [ ]:
# Check whether the expected columns exist
required_columns = [
    "StationId", "Datetime",
    "PM2.5", "PM10", "SO2", "NOx", "NH3", "CO", "O3",
    "AQI"
]

missing_columns = [
    col for col in required_columns
    if col not in station_df.columns
]

print("Missing required columns:", missing_columns)

# Inspect missing values in the relevant columns
available_columns = [
    col for col in required_columns
    if col in station_df.columns
]

print("\nMissing-value counts:")
display(station_df[available_columns].isna().sum())

# Check station and timestamp coverage
if "StationId" in station_df.columns:
    print("\nNumber of stations:", station_df["StationId"].nunique())

if "Datetime" in station_df.columns:
    print("Date range:")
    print(station_df["Datetime"].min(), "to", station_df["Datetime"].max())

Missing required columns: []

Missing-value counts:


StationId          0
Datetime           0
PM2.5         647689
PM10         1119252
SO2           742737
NOx           490808
NH3          1236618
CO            499302
O3            725973
AQI           570190
dtype: int64


Number of stations: 110
Date range:
2015-01-01 01:00:00 to 2020-07-01 00:00:00


In [ ]:
# 1. Check dataset size and duplicate records
print("Total rows:", len(station_df))
print("Exact duplicate rows:", station_df.duplicated().sum())

# 2. Convert Datetime to datetime format
station_df["Datetime"] = pd.to_datetime(
    station_df["Datetime"],
    errors="coerce"
)

print("\nInvalid timestamps:", station_df["Datetime"].isna().sum())

# 3. Check duplicate station-timestamp combinations
duplicate_station_times = station_df.duplicated(
    subset=["StationId", "Datetime"]
).sum()

print("Duplicate StationId-Datetime pairs:", duplicate_station_times)

# 4. Count recorded AQI values
print("\nAQI availability:")
print("Recorded AQI:", station_df["AQI"].notna().sum())
print("Missing AQI:", station_df["AQI"].isna().sum())

# 5. AQI distribution and extreme values
print("\nAQI descriptive statistics:")
display(station_df["AQI"].describe())

print("\nAQI values above 500:")
print((station_df["AQI"] > 500).sum())

# 6. Check hourly continuity for each station
station_df = station_df.sort_values(
    ["StationId", "Datetime"]
).copy()

station_df["time_gap"] = (
    station_df.groupby("StationId")["Datetime"].diff()
)

print("\nTime-gap distribution:")
display(station_df["time_gap"].value_counts().head(10))

print(
    "\nGaps longer than one hour:",
    (station_df["time_gap"] > pd.Timedelta(hours=1)).sum()
)

Total rows: 2589083
Exact duplicate rows: 0

Invalid timestamps: 0
Duplicate StationId-Datetime pairs: 0

AQI availability:
Recorded AQI: 2018893
Missing AQI: 570190

AQI descriptive statistics:


count    2.018893e+06
mean     1.801730e+02
std      1.404095e+02
min      5.000000e+00
25%      8.400000e+01
50%      1.310000e+02
75%      2.590000e+02
max      3.133000e+03
Name: AQI, dtype: float64


AQI values above 500:
43761

Time-gap distribution:


time_gap
0 days 01:00:00    2588973
Name: count, dtype: int64


Gaps longer than one hour: 0


In [ ]:
# Check whether station metadata is already included
print("Available columns:")
print(station_df.columns.tolist())

# Inspect a few rows
display(station_df.head())

# Check the number of observations per station
station_counts = (
    station_df.groupby("StationId")
    .size()
    .sort_values(ascending=False)
)

print("\nObservations per station:")
display(station_counts.describe())

print("\nFive stations with the most observations:")
display(station_counts.head())

print("\nFive stations with the fewest observations:")
display(station_counts.tail())

Available columns:
['StationId', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'time_gap']


,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,time_gap
0,AP001,2017-11-24 17:00:00,60.50,98.00,2.35,30.80,18.25,8.50,0.1,11.85,126.40,0.1,6.10,0.10,NaN,NaN,NaT
1,AP001,2017-11-24 18:00:00,65.50,111.25,2.70,24.20,15.07,9.77,0.1,13.17,117.12,0.1,6.25,0.15,NaN,NaN,0 days 01:00:00
2,AP001,2017-11-24 19:00:00,80.00,132.00,2.10,25.18,15.15,12.02,0.1,12.08,98.98,0.2,5.98,0.18,NaN,NaN,0 days 01:00:00
3,AP001,2017-11-24 20:00:00,81.50,133.25,1.95,16.25,10.23,11.58,0.1,10.47,112.20,0.2,6.72,0.10,NaN,NaN,0 days 01:00:00
4,AP001,2017-11-24 21:00:00,75.25,116.00,1.43,17.48,10.43,12.03,0.1,9.12,106.35,0.2,5.75,0.08,NaN,NaN,0 days 01:00:00



Observations per station:


count      110.000000
mean     23537.118182
std      14739.281331
min       2680.000000
25%       9441.250000
50%      21135.000000
75%      33186.500000
max      48192.000000
dtype: float64


Five stations with the most observations:


StationId
DL007    48192
DL033    48192
DL021    48192
DL013    48192
DL008    48192
dtype: int64


Five stations with the fewest observations:


StationId
BR005    2985
BR006    2863
HR013    2817
HR012    2817
MZ001    2680
dtype: int64

In [ ]:
# Check hourly coverage separately for every station

station_coverage = (
    station_df.groupby("StationId")
    .agg(
        first_record=("Datetime", "min"),
        last_record=("Datetime", "max"),
        total_records=("Datetime", "count"),
        unique_timestamps=("Datetime", "nunique")
    )
)

# Expected number of hourly observations within each station's date range
station_coverage["expected_hours"] = (
    (station_coverage["last_record"] - station_coverage["first_record"])
    / pd.Timedelta(hours=1)
    + 1
).astype(int)

station_coverage["missing_hours"] = (
    station_coverage["expected_hours"]
    - station_coverage["unique_timestamps"]
)

print("Stations checked:", len(station_coverage))
print("Stations with missing hours:", 
      (station_coverage["missing_hours"] > 0).sum())

print("\nStations with the most missing hours:")
display(
    station_coverage.sort_values(
        "missing_hours", ascending=False
    ).head(10)
)

Stations checked: 110
Stations with missing hours: 0

Stations with the most missing hours:


,first_record,last_record,total_records,unique_timestamps,expected_hours,missing_hours
StationId,,,,,,
AP001,2017-11-24 17:00:00,2020-07-01,22784,22784,22784,0
AP005,2016-07-01 12:00:00,2020-07-01,35053,35053,35053,0
AS001,2019-02-16 23:00:00,2020-07-01,12002,12002,12002,0
BR005,2020-02-27 16:00:00,2020-07-01,2985,2985,2985,0
BR006,2020-03-03 18:00:00,2020-07-01,2863,2863,2863,0
BR007,2015-06-01 15:00:00,2020-07-01,44554,44554,44554,0
BR008,2019-12-24 18:00:00,2020-07-01,4543,4543,4543,0
BR009,2019-12-24 18:00:00,2020-07-01,4543,4543,4543,0
BR010,2019-12-24 18:00:00,2020-07-01,4543,4543,4543,0


In [ ]:


stations_df=pd.read_csv('../data/raw/stations.csv')

print("Dataset shape:", stations_df.shape)

print("\nColumns:")
print(stations_df.columns.tolist())

print("\nFirst five rows:")
display(stations_df.head())

print("\nMissing values:")
display(stations_df.isna().sum())

Dataset shape: (230, 5)

Columns:
['StationId', 'StationName', 'City', 'State', 'Status']

First five rows:


,StationId,StationName,City,State,Status
0,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active
1,AP002,"Anand Kala Kshetram, Rajamahendravaram - APPCB",Rajamahendravaram,Andhra Pradesh,NaN
2,AP003,"Tirumala, Tirupati - APPCB",Tirupati,Andhra Pradesh,NaN
3,AP004,"PWD Grounds, Vijayawada - APPCB",Vijayawada,Andhra Pradesh,NaN
4,AP005,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active



Missing values:


StationId       0
StationName     0
City            0
State           0
Status         97
dtype: int64

In [ ]:
# 1. Check for duplicate station IDs in the metadata
print(
    "Duplicate StationIds in metadata:",
    stations_df["StationId"].duplicated().sum()
)

# 2. Check how many hourly stations have matching metadata
hourly_station_ids = set(station_df["StationId"].unique())
metadata_station_ids = set(stations_df["StationId"].unique())

matched_ids = hourly_station_ids & metadata_station_ids
unmatched_ids = hourly_station_ids - metadata_station_ids

print("\nUnique stations in station_hour:", len(hourly_station_ids))
print("Unique stations in metadata:", len(metadata_station_ids))
print("Matched stations:", len(matched_ids))
print("Unmatched hourly stations:", len(unmatched_ids))

if unmatched_ids:
    print("Unmatched StationIds:", sorted(unmatched_ids))

# 3. Check which cities are represented in the hourly dataset
hourly_metadata = stations_df[
    stations_df["StationId"].isin(hourly_station_ids)
].copy()

print("\nCities represented in station_hour:")
display(
    hourly_metadata.groupby("City")["StationId"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("number_of_stations")
)

# 4. Check whether any station ID maps to multiple cities
cities_per_station = stations_df.groupby("StationId")["City"].nunique()

print(
    "\nStationIds associated with multiple cities:",
    (cities_per_station > 1).sum()
)

Duplicate StationIds in metadata: 0

Unique stations in station_hour: 110
Unique stations in metadata: 230
Matched stations: 110
Unmatched hourly stations: 0

Cities represented in station_hour:


,number_of_stations
City,
Delhi,38
Bengaluru,10
Mumbai,10
Kolkata,7
Hyderabad,6
Patna,6
Lucknow,5
Gurugram,4
Chennai,4



StationIds associated with multiple cities: 0


In [ ]:
# Select the station metadata we need
station_metadata = stations_df[
    ["StationId", "City", "State"]
].copy()

# Merge city and state into hourly observations
station_aqi_df = station_df.merge(
    station_metadata,
    on="StationId",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Verify the merge
print("Original hourly rows:", len(station_df))
print("Rows after merge:", len(station_aqi_df))

print("\nMetadata merge results:")
print(station_aqi_df["_merge"].value_counts())

# Remove the merge indicator after verification
station_aqi_df.drop(columns="_merge", inplace=True)

print("\nDataset shape:", station_aqi_df.shape)
print("\nSample:")
display(
    station_aqi_df[
        ["StationId", "City", "State", "Datetime",
         "PM2.5", "PM10", "AQI", "AQI_Bucket"]
    ].head()
)

Original hourly rows: 2589083
Rows after merge: 2589083

Metadata merge results:
_merge
both          2589083
left_only           0
right_only          0
Name: count, dtype: int64

Dataset shape: (2589083, 19)

Sample:


,StationId,City,State,Datetime,PM2.5,PM10,AQI,AQI_Bucket
0,AP001,Amaravati,Andhra Pradesh,2017-11-24 17:00:00,60.50,98.00,NaN,NaN
1,AP001,Amaravati,Andhra Pradesh,2017-11-24 18:00:00,65.50,111.25,NaN,NaN
2,AP001,Amaravati,Andhra Pradesh,2017-11-24 19:00:00,80.00,132.00,NaN,NaN
3,AP001,Amaravati,Andhra Pradesh,2017-11-24 20:00:00,81.50,133.25,NaN,NaN
4,AP001,Amaravati,Andhra Pradesh,2017-11-24 21:00:00,75.25,116.00,NaN,NaN


In [ ]:
# Pollutants that use a 24-hour rolling average
average_pollutants = [
    "PM2.5", "PM10", "SO2", "NOx", "NH3"
]

# Pollutants that use an 8-hour rolling maximum
maximum_pollutants = ["CO", "O3"]

# Ensure observations are in chronological order within each station
station_aqi_df = station_aqi_df.sort_values(
    ["StationId", "Datetime"]
).copy()

# Calculate 24-hour rolling averages
for pollutant in average_pollutants:

    station_aqi_df[f"{pollutant}_24h_avg"] = (
        station_aqi_df
        .groupby("StationId")[pollutant]
        .transform(
            lambda s: s.rolling(
                window=24,
                min_periods=16
            ).mean()
        )
    )

# Calculate 8-hour rolling maximums
for pollutant in maximum_pollutants:

    station_aqi_df[f"{pollutant}_8h_max"] = (
        station_aqi_df
        .groupby("StationId")[pollutant]
        .transform(
            lambda s: s.rolling(
                window=8,
                min_periods=1
            ).max()
        )
    )

print("Rolling measurements calculated.")

print("\nFirst five rows:")
display(
    station_aqi_df[
        [
            "StationId",
            "Datetime",
            "PM2.5",
            "PM2.5_24h_avg",
            "PM10",
            "PM10_24h_avg",
            "CO",
            "CO_8h_max",
            "O3",
            "O3_8h_max"
        ]
    ].head()
)

Rolling measurements calculated.

First five rows:


,StationId,Datetime,PM2.5,PM2.5_24h_avg,PM10,PM10_24h_avg,CO,CO_8h_max,O3,O3_8h_max
0,AP001,2017-11-24 17:00:00,60.50,NaN,98.00,NaN,0.1,0.1,126.40,126.4
1,AP001,2017-11-24 18:00:00,65.50,NaN,111.25,NaN,0.1,0.1,117.12,126.4
2,AP001,2017-11-24 19:00:00,80.00,NaN,132.00,NaN,0.1,0.1,98.98,126.4
3,AP001,2017-11-24 20:00:00,81.50,NaN,133.25,NaN,0.1,0.1,112.20,126.4
4,AP001,2017-11-24 21:00:00,75.25,NaN,116.00,NaN,0.1,0.1,106.35,126.4


In [ ]:
rolling_columns = (
    [f"{p}_24h_avg" for p in average_pollutants]
    + [f"{p}_8h_max" for p in maximum_pollutants]
)

print("Available rolling measurements:")
display(
    station_aqi_df[rolling_columns]
    .notna()
    .sum()
    .to_frame("valid_rows")
)

print("\nPercentage available:")
display(
    (
        station_aqi_df[rolling_columns].notna().mean() * 100
    ).round(2).to_frame("availability_percent")
)

Available rolling measurements:


,valid_rows
PM2.5_24h_avg,1930066
PM10_24h_avg,1463183
SO2_24h_avg,1845088
NOx_24h_avg,2087458
NH3_24h_avg,1344862
CO_8h_max,2201453
O3_8h_max,1928356



Percentage available:


,availability_percent
PM2.5_24h_avg,74.55
PM10_24h_avg,56.51
SO2_24h_avg,71.26
NOx_24h_avg,80.63
NH3_24h_avg,51.94
CO_8h_max,85.03
O3_8h_max,74.48


In [ ]:
import numpy as np
import pandas as pd


def get_PM25_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 30:
        return x * 50 / 30
    elif x <= 60:
        return 50 + (x - 30) * 50 / 30
    elif x <= 90:
        return 100 + (x - 60) * 100 / 30
    elif x <= 120:
        return 200 + (x - 90) * 100 / 30
    elif x <= 250:
        return 300 + (x - 120) * 100 / 130
    else:
        return 400 + (x - 250) * 100 / 130


def get_PM10_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 100:
        return x
    elif x <= 250:
        return 100 + (x - 100) * 100 / 150
    elif x <= 350:
        return 200 + (x - 250)
    elif x <= 430:
        return 300 + (x - 350) * 100 / 80
    else:
        return 400 + (x - 430) * 100 / 80


def get_SO2_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 380:
        return 100 + (x - 80) * 100 / 300
    elif x <= 800:
        return 200 + (x - 380) * 100 / 420
    elif x <= 1600:
        return 300 + (x - 800) * 100 / 800
    else:
        return 400 + (x - 1600) * 100 / 800


def get_NOx_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 180:
        return 100 + (x - 80)
    elif x <= 280:
        return 200 + (x - 180)
    elif x <= 400:
        return 300 + (x - 280) * 100 / 120
    else:
        return 400 + (x - 400) * 100 / 120


def get_NH3_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 200:
        return x * 50 / 200
    elif x <= 400:
        return 50 + (x - 200) * 50 / 200
    elif x <= 800:
        return 100 + (x - 400) * 100 / 400
    elif x <= 1200:
        return 200 + (x - 800) * 100 / 400
    elif x <= 1800:
        return 300 + (x - 1200) * 100 / 600
    else:
        return 400 + (x - 1800) * 100 / 600


def get_CO_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 1:
        return x * 50
    elif x <= 2:
        return 50 + (x - 1) * 50
    elif x <= 10:
        return 100 + (x - 2) * 100 / 8
    elif x <= 17:
        return 200 + (x - 10) * 100 / 7
    elif x <= 34:
        return 300 + (x - 17) * 100 / 17
    else:
        return 400 + (x - 34) * 100 / 17


def get_O3_subindex(x):
    if pd.isna(x):
        return np.nan
    if x <= 50:
        return x
    elif x <= 100:
        return 50 + (x - 50)
    elif x <= 168:
        return 100 + (x - 100) * 100 / 68
    elif x <= 208:
        return 200 + (x - 168) * 100 / 40
    elif x <= 748:
        return 300 + (x - 208) * 100 / 540
    else:
        return 400 + (x - 748) * 100 / 540

In [ ]:
# Map each rolling measurement to its sub-index function
subindex_config = {
    "PM2.5_24h_avg": ("PM2.5_SubIndex", get_PM25_subindex),
    "PM10_24h_avg": ("PM10_SubIndex", get_PM10_subindex),
    "SO2_24h_avg": ("SO2_SubIndex", get_SO2_subindex),
    "NOx_24h_avg": ("NOx_SubIndex", get_NOx_subindex),
    "NH3_24h_avg": ("NH3_SubIndex", get_NH3_subindex),
    "CO_8h_max": ("CO_SubIndex", get_CO_subindex),
    "O3_8h_max": ("O3_SubIndex", get_O3_subindex),
}

for input_col, (output_col, function) in subindex_config.items():
    station_aqi_df[output_col] = station_aqi_df[input_col].apply(function)

print("Sub-index calculation completed.")

Sub-index calculation completed.


In [ ]:
subindex_columns = [
    "PM2.5_SubIndex",
    "PM10_SubIndex",
    "SO2_SubIndex",
    "NOx_SubIndex",
    "NH3_SubIndex",
    "CO_SubIndex",
    "O3_SubIndex",
]

# Check the number and percentage of available sub-indexes
subindex_summary = pd.DataFrame({
    "Available": station_aqi_df[subindex_columns].notna().sum(),
    "Missing": station_aqi_df[subindex_columns].isna().sum(),
    "Min": station_aqi_df[subindex_columns].min(),
    "Median": station_aqi_df[subindex_columns].median(),
    "Max": station_aqi_df[subindex_columns].max(),
})

subindex_summary["Available_Percent"] = (
    subindex_summary["Available"] / len(station_aqi_df) * 100
).round(2)

display(subindex_summary)

,Available,Missing,Min,Median,Max,Available_Percent
PM2.5_SubIndex,1930066,659017,0.262500,94.444444,976.923077,74.55
PM10_SubIndex,1463183,1125900,0.027368,115.760870,1112.500000,56.51
SO2_SubIndex,1845088,743995,0.055882,11.309524,132.253704,71.26
NOx_SubIndex,2087458,501625,0.000000,33.229687,432.744271,80.63
NH3_SubIndex,1344862,1244221,0.005000,5.929479,103.976562,51.94
CO_SubIndex,2201453,387630,0.000000,59.500000,3132.764706,85.03
O3_SubIndex,1928356,660727,0.010000,43.650000,446.111111,74.48


In [ ]:
# Missing rolling measurements must produce missing sub-indexes
missing_value_check = {}

for input_col, (output_col, function) in subindex_config.items():
    missing_value_check[output_col] = (
        station_aqi_df.loc[
            station_aqi_df[input_col].isna(), output_col
        ].notna().sum()
    )

print("Unexpected non-missing sub-indexes for missing inputs:")
print(missing_value_check)

Unexpected non-missing sub-indexes for missing inputs:
{'PM2.5_SubIndex': np.int64(0), 'PM10_SubIndex': np.int64(0), 'SO2_SubIndex': np.int64(0), 'NOx_SubIndex': np.int64(0), 'NH3_SubIndex': np.int64(0), 'CO_SubIndex': np.int64(0), 'O3_SubIndex': np.int64(0)}


In [ ]:
# Check important breakpoint values for each pollutant.
# These tests validate the function implementations, not the dataset.

breakpoint_tests = {
    "PM2.5": (
        get_PM25_subindex,
        [0, 30, 60, 90, 120, 250],
    ),
    "PM10": (
        get_PM10_subindex,
        [0, 50, 100, 250, 350, 430],
    ),
    "SO2": (
        get_SO2_subindex,
        [0, 40, 80, 380, 800, 1600],
    ),
    "NOx": (
        get_NOx_subindex,
        [0, 40, 80, 180, 280, 400],
    ),
    "NH3": (
        get_NH3_subindex,
        [0, 200, 400, 800, 1200, 1800],
    ),
    "CO": (
        get_CO_subindex,
        [0, 1, 2, 10, 17, 34],
    ),
    "O3": (
        get_O3_subindex,
        [0, 50, 100, 168, 208, 748],
    ),
}

for pollutant, (function, values) in breakpoint_tests.items():
    print(f"\n{pollutant}")

    for value in values:
        print(f"  Concentration: {value:>6} -> Sub-index: {function(value):.2f}")


PM2.5
  Concentration:      0 -> Sub-index: 0.00
  Concentration:     30 -> Sub-index: 50.00
  Concentration:     60 -> Sub-index: 100.00
  Concentration:     90 -> Sub-index: 200.00
  Concentration:    120 -> Sub-index: 300.00
  Concentration:    250 -> Sub-index: 400.00

PM10
  Concentration:      0 -> Sub-index: 0.00
  Concentration:     50 -> Sub-index: 50.00
  Concentration:    100 -> Sub-index: 100.00
  Concentration:    250 -> Sub-index: 200.00
  Concentration:    350 -> Sub-index: 300.00
  Concentration:    430 -> Sub-index: 400.00

SO2
  Concentration:      0 -> Sub-index: 0.00
  Concentration:     40 -> Sub-index: 50.00
  Concentration:     80 -> Sub-index: 100.00
  Concentration:    380 -> Sub-index: 200.00
  Concentration:    800 -> Sub-index: 300.00
  Concentration:   1600 -> Sub-index: 400.00

NOx
  Concentration:      0 -> Sub-index: 0.00
  Concentration:     40 -> Sub-index: 50.00
  Concentration:     80 -> Sub-index: 100.00
  Concentration:    180 -> Sub-index: 200.00

In [ ]:
# List the seven pollutant sub-index columns
subindex_columns = [
    "PM2.5_SubIndex",
    "PM10_SubIndex",
    "SO2_SubIndex",
    "NOx_SubIndex",
    "NH3_SubIndex",
    "CO_SubIndex",
    "O3_SubIndex",
]

# Count how many pollutant sub-indexes are positive
station_aqi_df["Checks"] = (
    station_aqi_df[subindex_columns] > 0
).sum(axis=1)

# Calculate the maximum available sub-index
station_aqi_df["AQI_calculated"] = (
    station_aqi_df[subindex_columns].max(axis=1).round()
)

# Require at least one particulate-matter sub-index to be positive
pm_columns = ["PM2.5_SubIndex", "PM10_SubIndex"]

pm_available = (
    station_aqi_df[pm_columns] > 0
).any(axis=1)

station_aqi_df.loc[
    ~pm_available, "AQI_calculated"
] = np.nan

# Require at least three positive pollutant sub-indexes
station_aqi_df.loc[
    station_aqi_df["Checks"] < 3, "AQI_calculated"
] = np.nan

print("Reconstructed AQI calculation completed.")

Reconstructed AQI calculation completed.


In [ ]:
# Summarize reconstructed AQI availability
total_rows = len(station_aqi_df)
available_aqi = station_aqi_df["AQI_calculated"].notna().sum()
missing_aqi = station_aqi_df["AQI_calculated"].isna().sum()

print(f"Total station-hour rows: {total_rows:,}")
print(f"Reconstructed AQI available: {available_aqi:,}")
print(f"Reconstructed AQI missing: {missing_aqi:,}")
print(f"Availability: {available_aqi / total_rows * 100:.2f}%")

display(station_aqi_df["AQI_calculated"].describe())

Total station-hour rows: 2,589,083
Reconstructed AQI available: 2,017,972
Reconstructed AQI missing: 571,111
Availability: 77.94%


count    2.017972e+06
mean     1.801687e+02
std      1.403840e+02
min      5.000000e+00
25%      8.400000e+01
50%      1.310000e+02
75%      2.590000e+02
max      3.133000e+03
Name: AQI_calculated, dtype: float64

In [ ]:
# Compare reconstructed AQI with the original AQI
comparison_df = station_aqi_df[
    ["AQI", "AQI_calculated"]
].dropna().copy()

comparison_df["Difference"] = (
    comparison_df["AQI"] - comparison_df["AQI_calculated"]
)

total_compared = len(comparison_df)

exact_matches = (
    comparison_df["AQI"] == comparison_df["AQI_calculated"]
).sum()

print(f"Rows compared: {total_compared:,}")
print(f"Exact matches: {exact_matches:,}")
print(f"Mismatches: {total_compared - exact_matches:,}")
print(f"Exact match rate: {exact_matches / total_compared * 100:.2f}%")

print("\nDifference summary:")
display(comparison_df["Difference"].describe())

Rows compared: 2,017,972
Exact matches: 2,015,133
Mismatches: 2,839
Exact match rate: 99.86%

Difference summary:


count    2.017972e+06
mean     2.479370e-02
std      1.194327e+00
min     -1.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.500000e+01
Name: Difference, dtype: float64

In [ ]:
# Count the different AQI discrepancies
print("Difference frequencies:")
display(
    comparison_df["Difference"]
    .value_counts()
    .sort_index()
    .to_frame("Count")
    .tail(20)
)

# Inspect the largest absolute differences
comparison_df["Absolute_Difference"] = (
    comparison_df["Difference"].abs()
)

largest_mismatches = comparison_df.loc[
    comparison_df["Absolute_Difference"] > 0
].sort_values(
    "Absolute_Difference",
    ascending=False
)

print("\nLargest mismatches:")
display(largest_mismatches.head(20))

Difference frequencies:


,Count
Difference,
46.0,5
47.0,8
48.0,4
49.0,4
50.0,3
51.0,8
52.0,4
53.0,3
54.0,3



Largest mismatches:


,AQI,AQI_calculated,Difference,Absolute_Difference
917793,494.0,429.0,65.0,65.0
917794,494.0,429.0,65.0,65.0
937250,466.0,401.0,65.0,65.0
937251,466.0,401.0,65.0,65.0
922261,501.0,436.0,65.0,65.0
922260,501.0,436.0,65.0,65.0
937252,466.0,401.0,65.0,65.0
937253,466.0,401.0,65.0,65.0
937254,466.0,401.0,65.0,65.0
937255,466.0,401.0,65.0,65.0


In [ ]:
# Get the original records corresponding to the largest mismatches
mismatch_indices = largest_mismatches.head(20).index

diagnostic_columns = [
    "StationId",
    "City",
    "Datetime",
    "AQI",
    "AQI_calculated",
    "PM2.5_24h_avg",
    "PM10_24h_avg",
    "SO2_24h_avg",
    "NOx_24h_avg",
    "NH3_24h_avg",
    "CO_8h_max",
    "O3_8h_max",
    "PM2.5_SubIndex",
    "PM10_SubIndex",
    "SO2_SubIndex",
    "NOx_SubIndex",
    "NH3_SubIndex",
    "CO_SubIndex",
    "O3_SubIndex",
]

mismatch_details = station_aqi_df.loc[
    mismatch_indices, diagnostic_columns
].copy()

# Add the difference from comparison_df
mismatch_details["Difference"] = comparison_df.loc[
    mismatch_indices, "Difference"
]

display(mismatch_details)

,StationId,City,Datetime,AQI,AQI_calculated,PM2.5_24h_avg,PM10_24h_avg,SO2_24h_avg,NOx_24h_avg,NH3_24h_avg,CO_8h_max,O3_8h_max,PM2.5_SubIndex,PM10_SubIndex,SO2_SubIndex,NOx_SubIndex,NH3_SubIndex,CO_SubIndex,O3_SubIndex,Difference
917793,DL028,Delhi,2015-11-23 09:00:00,494.0,429.0,282.514167,282.514167,NaN,NaN,40.061250,5.16,905.17,425.010897,232.514167,NaN,NaN,10.015313,139.500,429.105556,65.0
917794,DL028,Delhi,2015-11-23 10:00:00,494.0,429.0,278.930833,278.930833,NaN,NaN,40.456667,3.16,905.17,422.254487,228.930833,NaN,NaN,10.114167,114.500,429.105556,65.0
937250,DL028,Delhi,2018-02-11 02:00:00,466.0,401.0,231.166667,231.166667,18.010833,118.770952,34.280000,8.22,754.00,385.512821,187.444444,22.513542,138.770952,8.570000,177.750,401.111111,65.0
937251,DL028,Delhi,2018-02-11 03:00:00,466.0,401.0,225.708333,225.708333,17.909167,117.773810,34.426667,8.22,754.00,381.314103,183.805556,22.386458,137.773810,8.606667,177.750,401.111111,65.0
922261,DL028,Delhi,2016-05-27 13:00:00,501.0,436.0,153.441250,153.441250,20.339545,100.400833,51.973750,0.97,944.00,325.724038,135.627500,25.424432,120.400833,12.993438,48.500,436.296296,65.0
922260,DL028,Delhi,2016-05-27 12:00:00,501.0,436.0,153.878750,153.878750,20.181364,100.041667,52.111667,1.16,944.00,326.060577,135.919167,25.226705,120.041667,13.027917,58.000,436.296296,65.0
937252,DL028,Delhi,2018-02-11 04:00:00,466.0,401.0,217.833333,217.833333,18.025417,121.126000,35.934583,8.22,754.00,375.256410,178.555556,22.531771,141.126000,8.983646,177.750,401.111111,65.0
937253,DL028,Delhi,2018-02-11 05:00:00,466.0,401.0,209.791667,209.791667,17.935833,142.931500,37.868333,8.22,754.00,369.070513,173.194444,22.419792,162.931500,9.467083,177.750,401.111111,65.0
937254,DL028,Delhi,2018-02-11 06:00:00,466.0,401.0,201.791667,201.791667,17.704167,160.979500,39.322500,6.69,754.00,362.916667,167.861111,22.130208,180.979500,9.830625,158.625,401.111111,65.0
937255,DL028,Delhi,2018-02-11 07:00:00,466.0,401.0,194.833333,194.833333,18.014583,177.213500,41.208750,6.69,754.00,357.564103,163.222222,22.518229,197.213500,10.302188,158.625,401.111111,65.0


## AQI Reconstruction Audit — Findings

### Objective
Reconstruct station-hour AQI values from pollutant measurements to
investigate the consistency of the original AQI data.

### Method
- Calculated rolling pollutant averages and maxima.
- Applied pollutant-specific sub-index formulas.
- Reconstructed AQI using the maximum available sub-index, subject
  to the minimum pollutant availability conditions.
- Compared reconstructed AQI with the original AQI.

### Results
- Total station-hour records: 2,589,083
- Reconstructed AQI available: 2,017,972
- Records compared with original AQI: 2,017,972
- Exact matches: 2,015,133
- Mismatches: 2,839
- Exact match rate: 99.86%

### Limitations
- Some discrepancies are substantial and require further investigation.
- The reconstruction does not establish that every original AQI value
  is independently correct.
- The implementation does not fully reproduce every official CPCB
  methodological detail, including the special ozone averaging rule.
- Reconstructed AQI is used for data auditing, not as a model input.

### Decision
The audit indicates strong agreement between reconstructed and original
AQI values. Record the remaining discrepancies and proceed to the
classification modelling workflow, revisiting them if necessary.

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
display(df["AQI_Bucket"].value_counts(dropna=False))

print("\nMissing target values:")
print(df["AQI_Bucket"].isna().sum())

Dataset shape: (707875, 17)

Columns:
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'Expected_AQI_Bucket']

Target distribution:


AQI_Bucket
Moderate        198991
Satisfactory    189434
NaN             129080
Poor             66654
Very Poor        57455
Good             38611
Severe           27650
Name: count, dtype: int64


Missing target values:
129080


In [ ]:
# Define the initial pollutant features
pollutant_features = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
]

target = "AQI_Bucket"

# Keep rows with known target labels
model_df = df.loc[df[target].notna()].copy()

# Separate model inputs and target
X = model_df[pollutant_features].copy()
y = model_df[target].copy()

print("Original dataset:", df.shape)
print("Rows with known labels:", len(model_df))
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
display(y.value_counts())

print("\nMissing values in features:")
display(X.isna().sum().sort_values(ascending=False))

print("\nTarget classes:", sorted(y.unique()))

Original dataset: (707875, 17)
Rows with known labels: 578795
Feature matrix shape: (578795, 10)
Target shape: (578795,)

Target distribution:


AQI_Bucket
Moderate        198991
Satisfactory    189434
Poor             66654
Very Poor        57455
Good             38611
Severe           27650
Name: count, dtype: int64


Missing values in features:


PM10       178863
NH3        163994
Benzene     96365
NOx         51906
O3          33372
SO2         33058
PM2.5       31035
CO          29146
NO2         22267
NO          21481
dtype: int64


Target classes: ['Good', 'Moderate', 'Poor', 'Satisfactory', 'Severe', 'Very Poor']


In [ ]:
# Ensure timestamps are in datetime format
model_df["Datetime"] = pd.to_datetime(model_df["Datetime"])

# Get unique timestamps in chronological order
unique_times = np.sort(model_df["Datetime"].unique())

# Set the cutoff at approximately 80% of unique timestamps
cutoff_index = int(len(unique_times) * 0.8)
cutoff_time = unique_times[cutoff_index]

# Split earlier observations for training and later observations for testing
train_mask = model_df["Datetime"] < cutoff_time
test_mask = model_df["Datetime"] >= cutoff_time

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

# Inspect the split
print("Cutoff timestamp:", cutoff_time)

print("\nTraining observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining date range:")
print(
    model_df.loc[train_mask, "Datetime"].min(),
    "to",
    model_df.loc[train_mask, "Datetime"].max()
)

print("\nTesting date range:")
print(
    model_df.loc[test_mask, "Datetime"].min(),
    "to",
    model_df.loc[test_mask, "Datetime"].max()
)

print("\nTraining class distribution:")
display(y_train.value_counts())

print("\nTesting class distribution:")
display(y_test.value_counts())

Cutoff timestamp: 2019-05-26T13:00:00.000000

Training observations: 372595
Testing observations: 206200

Training date range:
2015-01-01 16:00:00 to 2019-05-26 12:00:00

Testing date range:
2019-05-26 13:00:00 to 2020-07-01 00:00:00

Training class distribution:


AQI_Bucket
Moderate        133454
Satisfactory    102197
Poor             51449
Very Poor        47492
Severe           23191
Good             14812
Name: count, dtype: int64


Testing class distribution:


AQI_Bucket
Satisfactory    87237
Moderate        65537
Good            23799
Poor            15205
Very Poor        9963
Severe           4459
Name: count, dtype: int64

In [ ]:
model_df = df.loc[df[target].notna()].copy()
X = model_df[feature_cols]
y = model_df[target]

NameError: name 'df' is not defined